# ReAct from scratch

Every agent framework wraps the same loop:

```
while not done:
    thought, action, args = llm(history)
    obs = tools[action](**args)
    history += [thought, action, obs]
```

Yao et al. (2022) called this **ReAct** — *Reason + Act*. Below is a faithful implementation in 30 lines using the shared task tools.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / '03-agentic-frameworks'))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
from task import TOOLS, get_question, run_evaluation
from eval import react_solve, REACT_SYS
print('tools:', list(TOOLS))

## The prompt that drives the loop

The model is asked to output one of two shapes per turn:

```
Thought: <reasoning>
Action: <tool_name>
Action Input: <json args>
```

or, when ready to finish:

```
Thought: <reasoning>
Final Answer: <citation-rich answer>
```

Everything else is a parse error.

In [ ]:
print(REACT_SYS)

## Run on a demo question

In [ ]:
trace = react_solve(get_question('q01'))
for step in trace.steps:
    if step.role == 'tool_call':
        print(f'-> {step.name}({step.arguments})')
        print(f'   = {step.output_summary[:120]}')
    else:
        print('ANSWER:', step.content)

## The two failure modes the frameworks paper over

1. **Parse errors** — the model returns prose instead of `Action: foo`. The scratch loop just stops; production frameworks retry with a self-correction prompt.
2. **Infinite loops** — the model never emits `Final Answer:`. The scratch loop sets `MAX_STEPS=6`; production frameworks add a budget timer.

Knowing these failure modes is the value of writing this once. Every framework downstream just wraps the same loop with retries, structured tool calls, observability, and checkpoints.

In [ ]:
result = run_evaluation(react_solve, framework='react-from-scratch')
import json
print(json.dumps(result['averages'], indent=2))